# RoentGen on CheXpert: reproducibility notebook (Kaggle)

IE643, Team NeuroSight. This notebook lets a grader run the pipeline end to end at small scale:
1. **Stage 1**: build the manifest and patient-level splits from CheXpert-v1.0-small.
2. **Stage 1b/2 smoke test**: cache latents for 64 images and run 20 training steps of run C (full U-Net @256).
3. **Sampling**: generate pathology-conditioned CXRs from the released run-C weights (C@15k, g = 4) and score them with the judge.
4. **Tables**: rebuild the result tables and the Stage 5 figure from the committed `results/*.json`.

**Setup:** GPU accelerator (T4 x2 preferred; the code uses bf16, which the T4 emulates, so it is slower than on the RTX 4070 it was developed on) and Internet on. Add these inputs:
- a CheXpert-v1.0-small dataset (any mirror; `train.csv` is found automatically under `/kaggle/input`)
- *optional, for step 3*: a dataset holding `trainable.safetensors` (run C, ckpt-15000) and `best.pt` (the judge, `artifacts/cls/real_f1.0_s0`)

In [ ]:
REPO_URL = "https://github.com/<owner>/IE643-RoentGen-CheXpert.git"   # <- set to the project repository
WEIGHTS_DIR = "/kaggle/input/roentgen-chexpert-weights"               # optional: trainable.safetensors + best.pt

!git clone -q $REPO_URL /kaggle/working/IE643-RoentGen-CheXpert
%cd /kaggle/working/IE643-RoentGen-CheXpert
# torch/torchvision come with the Kaggle image; pin the rest as in requirements.txt
!pip install -q diffusers==0.31.0 transformers==4.44.2 peft==0.13.2 accelerate==1.0.1 torchxrayvision==1.2.4 \
    torch-fidelity==0.3.0 open-clip-torch==2.24.0 ftfy scikit-multilearn==0.2.0

In [ ]:
import os, sys
ROOT = "/kaggle/working/IE643-RoentGen-CheXpert"
os.environ.update(IE643_ROOT=ROOT, CXR_DATA_ROOT="/kaggle/input", PYTHONPATH=f"{ROOT}/src",
                  PYTORCH_CUDA_ALLOC_CONF="expandable_segments:True")
sys.path.insert(0, f"{ROOT}/src")
from roentgen.config import find_chexpert_dir
print("CheXpert at", find_chexpert_dir())

## Stage 1: manifest, uncertainty policy, patient-level splits

In [ ]:
!python -m roentgen.chexpert --build
import json; m = json.load(open("artifacts/manifest_meta.json"))
print(m["n_images"]); print(m["positives"]["train"])

## Stage 1b + 2 smoke test: latent cache (64 images) and 20 steps of run C

In [ ]:
!python -m roentgen.cache_latents --size 256 --smoke
!python -m roentgen.train_lora --run C --smoke --out artifacts/runs/C_smoke

## Sampling from the released weights (C@15k, DPM++ 25 steps, g = 4)
Skipped if `WEIGHTS_DIR` is not attached. The judge's probabilities show that the prompted finding is present.

In [ ]:
from pathlib import Path
w = Path(WEIGHTS_DIR)
HAVE_WEIGHTS = (w / "trainable.safetensors").exists() and (w / "best.pt").exists()
if HAVE_WEIGHTS:
    for src, dst in [(w / "trainable.safetensors", "artifacts/runs/C/ckpt-15000/trainable.safetensors"),
                     (w / "best.pt", "artifacts/cls/real_f1.0_s0/best.pt")]:
        Path(dst).parent.mkdir(parents=True, exist_ok=True)
        if not Path(dst).exists():
            os.symlink(src, dst)
print("weights attached:", HAVE_WEIGHTS)

In [ ]:
if HAVE_WEIGHTS:
    import matplotlib.pyplot as plt, numpy as np, torch
    from roentgen.compare_runs import build_pipe
    from roentgen.config import TARGETS
    from roentgen.prompts import PHRASES
    from roentgen.train_classifier import load_judge, predict

    pipe, judge = build_pipe("C", 15000), load_judge()
    prompts = [f"frontal chest radiograph showing {PHRASES[t]}" for t in TARGETS] + ["normal frontal chest x-ray"]
    with torch.no_grad(), torch.autocast("cuda", dtype=torch.bfloat16):
        ims = pipe(prompts, height=256, width=256, num_inference_steps=25, guidance_scale=4.0,
                   generator=[torch.Generator("cuda").manual_seed(643 + i) for i in range(len(prompts))],
                   output_type="np").images
    u8 = (ims.mean(-1) * 255).round().astype(np.uint8)
    probs = predict(judge, u8)
    fig, axs = plt.subplots(1, len(prompts), figsize=(3 * len(prompts), 3.6))
    for ax, im, p, t in zip(axs, u8, probs, TARGETS + ["normal"]):
        ax.imshow(im, cmap="gray"); ax.axis("off")
        ax.set_title(t if t == "normal" else f"{t}\np = {p[TARGETS.index(t)]:.2f}", fontsize=9)
    plt.tight_layout(); plt.show()

## Result tables and the Stage 5 figure (from the committed results/*.json)

In [ ]:
!python -m roentgen.aggregate
from IPython.display import Image, Markdown, display
display(Image("docs/img/stage5_auroc_vs_size.png"))

## Interactive demo (optional)
`app/gradio_demo.py` needs the weights above. With `--share` it prints a public link:
```
!pip install -q gradio && python app/gradio_demo.py --share
```